In [ ]:
%pip -q install yfinance pandas numpy tqdm

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Eligible_T0_Securities_20260930121437.xlsx to Eligible_T0_Securities_20260930121437.xlsx


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving EQUITY_L.csv to EQUITY_L.csv


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving SME_EQUITY_L.csv to SME_EQUITY_L.csv


In [ ]:
import os
import pandas as pd

csv_path = "/content/EQUITY_L.csv"

if not os.path.exists(csv_path):
    raise FileNotFoundError(
        "EQUITY_L.csv not found. Upload it using the Files panel."
    )

raw = pd.read_csv(csv_path)

print("Loaded:", csv_path)
print("Columns:", list(raw.columns))
display(raw.head())

Loaded: /content/EQUITY_L.csv
Columns: ['SYMBOL', 'NAME OF COMPANY', ' SERIES', ' DATE OF LISTING', ' PAID UP VALUE', ' MARKET LOT', ' ISIN NUMBER', ' FACE VALUE']


,SYMBOL,NAME OF COMPANY,SERIES,DATE OF LISTING,PAID UP VALUE,MARKET LOT,ISIN NUMBER,FACE VALUE
0,20MICRONS,20 Microns Limited,EQ,06-OCT-2008,5,1,INE144J01027,5
1,21STCENMGM,21st Century Management Services Limited,EQ,03-MAY-1995,10,1,INE253B01015,10
2,360ONE,360 ONE WAM LIMITED,EQ,19-SEP-2019,1,1,INE466L01038,1
3,3BBLACKBIO,3B Blackbio Dx Limited,EQ,20-APR-2026,10,1,INE994E01018,10
4,3IINFOLTD,3i Infotech Limited,BE,22-OCT-2021,10,1,INE748C01038,10


In [ ]:
import re

def norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

cols = {norm(c): c for c in raw.columns}
symbol_col = next((cols[k] for k in ["symbol", "tradingsymbol", "ticker", "securitysymbol"] if k in cols), None)
if symbol_col is None:
    raise ValueError("Cannot identify the symbol column. Inspect raw.columns and set symbol_col manually.")
universe = raw[symbol_col].astype(str).str.strip().dropna().drop_duplicates().tolist()
universe = [s for s in universe if re.fullmatch(r"[A-Za-z0-9&._-]+", s) and s.lower() not in ["nan", "none"]]
print("Symbols found:", len(universe), "| Examples:", universe[:25])

MAX_STOCKS = 50  # Set to None to attempt all symbols; may take a long time.
START_DATE = "2015-01-01"
SKIP_EXISTING = True
PAUSE_SECONDS = 0.4
symbols_to_fetch = universe if MAX_STOCKS is None else universe[:MAX_STOCKS]
print("Symbols queued:", len(symbols_to_fetch))


Symbols found: 2623 | Examples: ['20MICRONS', '21STCENMGM', '360ONE', '3BBLACKBIO', '3IINFOLTD', '3MINDIA', '3PLAND', '5PAISA', '63MOONS', 'A2ZINFRA', 'AAATECH', 'AADHARHFC', 'AAKASH', 'AAREYDRUGS', 'AARNAV', 'AARON', 'AARTECH', 'AARTIDRUGS', 'AARTIIND', 'AARTIPHARM', 'AARTISURF', 'AARVI', 'AASTHA', 'AAVAS', 'ABAN']
Symbols queued: 50


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sqlite3
import time
import yfinance as yf
import pandas as pd
import numpy as np

In [ ]:
from tqdm.auto import tqdm

In [ ]:
from datetime import datetime, timezone

In [ ]:
DB_PATH = "/content/indian_stock_market.sqlite"
conn = sqlite3.connect(DB_PATH)
conn.execute("CREATE TABLE IF NOT EXISTS daily_prices (symbol TEXT NOT NULL, date TEXT NOT NULL, open REAL, high REAL, low REAL, close REAL, adj_close REAL, volume REAL, PRIMARY KEY(symbol,date))")
conn.execute("CREATE TABLE IF NOT EXISTS download_log (symbol TEXT PRIMARY KEY, status TEXT, rows_downloaded INTEGER, message TEXT, updated_at TEXT)")
conn.commit()

def downloaded_ok(symbol):
    row = conn.execute("SELECT status FROM download_log WHERE symbol=?", (symbol,)).fetchone()
    return row is not None and row[0] == "success"

def fetch_one(symbol):
    try:
        hist = yf.download(f"{symbol}.NS", start=START_DATE, auto_adjust=False, progress=False, threads=False, timeout=25)
        if hist is None or hist.empty:
            raise ValueError("No history returned; symbol may be unsupported by this data source.")
        if isinstance(hist.columns, pd.MultiIndex):
            hist.columns = hist.columns.get_level_values(0)
        hist = hist.rename(columns={"Open":"open", "High":"high", "Low":"low", "Close":"close", "Adj Close":"adj_close", "Volume":"volume"})
        for c in ["open", "high", "low", "close"]:
            if c not in hist.columns:
                raise ValueError("Missing OHLC columns: " + str(list(hist.columns)))
        if "adj_close" not in hist.columns: hist["adj_close"] = hist["close"]
        if "volume" not in hist.columns: hist["volume"] = np.nan
        hist = hist.reset_index()
        date_col = "Date" if "Date" in hist.columns else hist.columns[0]
        rows = []
        for _, r in hist.iterrows():
            dt = pd.to_datetime(r[date_col]).strftime("%Y-%m-%d")
            def val(k):
                v = r.get(k, np.nan)
                return None if pd.isna(v) else float(v)
            rows.append((symbol, dt, val("open"), val("high"), val("low"), val("close"), val("adj_close"), val("volume")))
        conn.executemany("INSERT OR REPLACE INTO daily_prices VALUES (?,?,?,?,?,?,?,?)", rows)
        conn.execute("INSERT OR REPLACE INTO download_log VALUES (?,?,?,?,?)", (symbol, "success", len(rows), "", datetime.now(timezone.utc).isoformat()))
        conn.commit()
        return True, len(rows), ""
    except Exception as e:
        conn.execute("INSERT OR REPLACE INTO download_log VALUES (?,?,?,?,?)", (symbol, "failed", 0, str(e)[:500], datetime.now(timezone.utc).isoformat()))
        conn.commit()
        return False, 0, str(e)

ok_count = fail_count = 0
for sym in tqdm(symbols_to_fetch, desc="Downloading"):
    if SKIP_EXISTING and downloaded_ok(sym):
        continue
    ok, n, msg = fetch_one(sym)
    if ok: ok_count += 1
    else: fail_count += 1
    time.sleep(PAUSE_SECONDS)

print("New successful downloads:", ok_count, "| New failures:", fail_count)
print("Database file:", DB_PATH)


Downloading:   0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
coverage = pd.read_sql_query("SELECT symbol, COUNT(*) AS daily_rows, MIN(date) AS first_date, MAX(date) AS last_date FROM daily_prices GROUP BY symbol ORDER BY symbol", conn)
failures = pd.read_sql_query("SELECT * FROM download_log WHERE status='failed' ORDER BY symbol", conn)
print("Successfully stored symbols:", len(coverage))
display(coverage.head(25))
print("Failed symbols:", len(failures))
display(failures.head(25))


In [ ]:
def analyze_group(g):
    g = g.sort_values("date").copy()
    base = g["adj_close"].where(g["adj_close"].notna(), g["close"]).astype(float)
    g["SMA50"] = base.rolling(50, min_periods=50).mean()
    g["SMA200"] = base.rolling(200, min_periods=200).mean()
    d = base.diff()
    ag = d.clip(lower=0).ewm(alpha=1/14, min_periods=14, adjust=False).mean()
    al = (-d.clip(upper=0)).ewm(alpha=1/14, min_periods=14, adjust=False).mean()
    g["RSI14"] = 100 - 100 / (1 + ag / al.replace(0, np.nan))
    g["Return20D_pct"] = base.pct_change(20) * 100
    valid = g.dropna(subset=["RSI14"])
    if valid.empty: return None
    r = valid.iloc[-1]
    score, reasons, risks = 0, [], []
    if pd.notna(r["SMA200"]):
        if base.loc[r.name] > r["SMA50"] > r["SMA200"]:
            score += 2; reasons.append("price > SMA50 > SMA200")
        elif base.loc[r.name] < r["SMA50"] < r["SMA200"]:
            score -= 2; risks.append("price < SMA50 < SMA200")
        elif r["SMA50"] > r["SMA200"]:
            score += 1; reasons.append("SMA50 > SMA200")
        else:
            score -= 1; risks.append("SMA50 < SMA200")
    else: risks.append("insufficient 200-day history")
    if 40 <= r["RSI14"] <= 65:
        score += 1; reasons.append("moderate RSI")
    elif r["RSI14"] > 70:
        score -= 1; risks.append("RSI above 70")
    elif r["RSI14"] < 30:
        risks.append("RSI below 30")
    if pd.notna(r["Return20D_pct"]):
        if r["Return20D_pct"] > 0:
            score += 1; reasons.append("positive 20-session return")
        else:
            score -= 1; risks.append("negative 20-session return")
    decision = "BUY WATCH" if score >= 3 else ("SELL WATCH" if score <= -2 else "HOLD / WAIT")
    return {"symbol": g["symbol"].iloc[-1], "date": r["date"], "close": r["close"], "RSI14": r["RSI14"], "SMA50": r["SMA50"], "SMA200": r["SMA200"], "Return20D_pct": r["Return20D_pct"], "score": score, "decision": decision, "reasons": "; ".join(reasons), "risks": "; ".join(risks)}

all_data = pd.read_sql_query("SELECT * FROM daily_prices ORDER BY symbol,date", conn)
results = []
for sym, group in tqdm(all_data.groupby("symbol"), desc="Scanning"):
    result = analyze_group(group)
    if result: results.append(result)
scan = pd.DataFrame(results)
if not scan.empty:
    display(scan.sort_values(["score", "Return20D_pct"], ascending=False).head(50))
    print("Stocks scanned:", len(scan))
else:
    print("No data to scan yet.")


In [ ]:
all_data.to_csv("/content/indian_stock_daily_prices.csv", index=False)
scan.to_csv("/content/indian_stock_scan_results.csv", index=False)
failures.to_csv("/content/indian_stock_download_failures.csv", index=False)
conn.commit()
conn.close()

from google.colab import files
files.download(DB_PATH)  # database contains all downloaded daily price rows
# Optional: download CSVs too by uncommenting these lines:
# files.download("/content/indian_stock_daily_prices.csv")
# files.download("/content/indian_stock_scan_results.csv")
# files.download("/content/indian_stock_download_failures.csv")


In [ ]:
import pandas as pd
from IPython.display import display

results = pd.read_csv("/content/indian_stock_scan_results.csv")

print("Total stocks in scan:", len(results))
display(results)

In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=results)